In [12]:
import smolagents
import sqlalchemy
import json

In [4]:
import sys
sys.path.append("..")

from app.adapters.repositories.database import get_db

In [75]:
session = get_db()

In [9]:
rows = session.execute(sqlalchemy.text("SELECT name, schema from app.templates where schema is not null"))
rows_mapping = {
    row[0]: row[1] for row in rows
}

In [55]:
rows_mapping_str = ""
for k, v in rows_mapping.items():
    rows_mapping_str += f"Category: {k}\nSchema: \n```json{v}```\n"

In [79]:
class MetadataQueryAgent(smolagents.Tool):
    name="metadata_query_agent"
    description=f"""
    # Instruction
    Generate PostgreSQL to answer user query for a given time range. Returns a string representation of the result.
    Beware that this tool's output is a string representation of the execution output.
    # Database
    It can use the following tables:
    Table: contents_data
    Columns:
        - id: VARCHAR
        - name: VARCHAR content type
        - title: VARCHAR
        - metadata_data: JSON
        - created_at: TIMESTAMP
        - updated_at: TIMESTAMP
    Each document metadata in the database is stored in JSON format. There are multiple metadatas in the database. Here is the list:
    {rows_mapping_str}
    # Guideline
    In order to answer user query, you need to generate a PostgreSQL query that will return the required data for the given time range.
    Use json_extract_path_text function to extract the data from the JSON document.
    The table contents is constructed as key-value table, where it looks up schema from the templates table, and use JSON query path extract to get desired value.
    You need to analyze query, find out which content types to use and extract the value from the JSON.
    # Example
    Task: Calculate total payment in shopping and foods in last 7 days
    Output:
        WITH last_seven_days AS (
            SELECT 
                json_extract_path_text(metadata_data, 'date') as date,
                json_extract_path_text(metadata_data, 'total_amount') as total_amount
            FROM 
                contents_data
            where created_at < now() - interval '7 days'
        ),
        filtered_data AS (
            SELECT 
                total_amount::numeric
            FROM 
                last_seven_days
        )
        SELECT 
            sum(total_amount)
        FROM 
            filtered_data
    """
    inputs = {
        "task": {
            "type": "string",
            "description": "The SQL query to execute."
        }
    }
    output_type = "string"
    def __init__(self, session):
        super().__init__()
        self.session = session
    def forward(self, task: str) -> str:
        with self.session as conn:
            query = conn.execute(sqlalchemy.text(task))
            result = query.fetchall()
            result_str = str(result)
            return result_str

In [80]:
metadata_query_agent = MetadataQueryAgent(session)

In [81]:
query = """
    SELECT MAX(total_amount::numeric)                                                                                
  FROM (                                                                                                           
      SELECT json_extract_path_text(metadata_data, 'total_amount') as total_amount                                 
      FROM contents_data                                                                                           
      WHERE name = 'receipts_and_bills > food_and_dining'                                                          
         OR name = 'receipts_and_bills > retail_and_shopping'                                                      
  ) AS extracted_amounts;  
"""
metadata_query_agent.forward(query)

"[(Decimal('3256000'),)]"

In [ ]:
from smolagents import CodeAgent, LiteLLMModel

model_id = "openai/qwen2.5-14b-instruct-mlx"
url = "https://api.deepinfra.com/v1/openai1"
api_key = "asd"
url = "https://generativelanguage.googleapis.com/v1beta/openai/"
api_key = ""
model_id = "openai/gemini-2.5-pro-exp-03-25"
model = LiteLLMModel(
    model_id=model_id, # This model is a bit weak for agentic behaviours though
    api_base=url, # replace with 127.0.0.1:11434 or remote open-ai compatible server if necessary
    api_key=api_key, # replace with API key if necessary
    # num_ctx=128_000, # ollama default is 2048 which will fail horribly. 8192 works for easy tasks, more is better. Check https://huggingface.co/spaces/NyxKrage/LLM-Model-VRAM-Calculator to calculate how much VRAM this will need for the selected model.
)

In [83]:
agent = CodeAgent(
    model=model,
    tools=[metadata_query_agent],
    name="agent_analyze_data",
    description="A tool that allows you to perform SQL queries on the table. Returns a string representation of the result.",
)

In [84]:
t = agent.run(task="Which highest bill was paid?", stream=True)
for chunk in t:
    if chunk is not None:
        print(chunk, end="")
    else:
        break
print("\n\n")

╭───────────────────────────────────────── New run - agent_analyze_data ──────────────────────────────────────────╮
│                                                                                                                 │
│ Which highest bill was paid?                                                                                    │
│                                                                                                                 │
╰─ LiteLLMModel - openai/gemini-2.5-pro-exp-03-25 ────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  Code:                                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Code parsing failed on line 1 due to: SyntaxError
Code:
      ^
Error: invalid syntax (<unknown>, line 1)

[Step 1: Duration 12.04 seconds| Input tokens: 11,600 | Output tokens: 1,312]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  result_data = [('3256000', 'Ao Quán', '18/07/2018', 'receipts_and_bills > food_and_dining')]                     
  highest_bill = result_data[0]                                                                                    
  amount = highest_bill[0]                                                                                         
  location = highest_bill[1]                                                                                       
  date = highest_bill[2]                                                                                           
                                                                                                                   
  # It's good practice to check if the currency is available, but the current query doesn't retrieve it.           
  # Assuming the currency might be important context, but based on the query result, I can only state the amount.  
  # Let's format the amount for better readability if possible.                                                    
  try:                                                                                                             
      formatted_amount = f"{int(amount):,}"                                                                        
  except ValueError:                                                                                               
      formatted_amount = amount # Fallback if it's not a simple integer string                                     
                                                                                                                   
  answer = f"The highest bill found was for {formatted_amount} at {location} on {date}."                           
  final_answer(answer)                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out - Final answer: The highest bill found was for 3,256,000 at Ao Quán on 18/07/2018.

[Step 3: Duration 6.14 seconds| Input tokens: 37,514 | Output tokens: 2,771]

ActionStep(model_input_messages=[{'role': <MessageRole.SYSTEM: 'system'>, 'content': [{'type': 'text', 'text': 'You are an expert assistant who can solve any task using code blobs. You will be given a task to solve as best you can.\nTo do so, you have been given access to a list of tools: these tools are basically Python functions which you can call with code.\nTo solve the task, you must plan forward to proceed in a series of steps, in a cycle of \'Thought:\', \'Code:\', and \'Observation:\' sequences.\n\nAt each step, in the \'Thought:\' sequence, you should first explain your reasoning towards solving the task and the tools that you want to use.\nThen in the \'Code:\' sequence, you should write the code in simple Python. The code sequence must end with \'<end_code>\' sequence.\nDuring each intermediate step, you can use \'print()\' to save whatever important information you will then need.\nThese print outputs will then appear in the \'Observation:\' field, which will be available a